# ViT-L dense-token — calibration puis balayage (Projet_mellifere)

Ce notebook **ne fait pas glisser des crops 224×224 à stride 112**. `ortho_annotator` a déjà un détecteur dense : un passage avant sur une fenêtre de 784² (par défaut) donne une grille de jetons de patch à ~10 cm de résolution au sol — c'est la heatmap, en un forward pass au lieu de centaines de crops. Le balayage encode maintenant **plusieurs fenêtres par passage** (`--embed-batch-size`) plutôt qu'une seule à la fois — sur GPU, batch=1 laisserait l'essentiel de la VRAM inutilisée quel que soit le GPU choisi.

**Déroulé en 2 étapes, avec une porte de décision entre les deux :**

1. **Étape A — calibration.** `prospect learn --model dinov3-vitl16 --rasters Maison` (un seul ortho, ~14 Go, celui qui a le plus d'espèces). Ça mesure le rappel/précision dense par espèce, comparable au tableau du README (mesuré en ViT-S/16 sur les 7 orthos). **Si ViT-L ne bat pas ces chiffres, le balayage complet ci-dessous n'apporte rien — ne pas le lancer.**
2. **Étape B — balayage.** Seulement si l'étape A montre un vrai gain : `prospect scan --mode dense --embed-batch-size N` sur les orthos choisis, puis export GPKG pour QGIS.

Rappels utiles (voir la conversation qui a produit ce notebook) :
- Le balayage dense est déjà **row-major séquentiel** (`prospect.py:scan_raster_dense`), cohérent avec le stockage en bandes des GeoTIFF — pas d'accès aléatoire à corriger.
- `Lotcorn`/`Leuvul` n'existent que sur l'ortho **Maison** ; `Daucar` est à 82% sur **TrailErable**. Pas besoin des 131 Go de `Dataset_Leo`/`Projet_mellifere` d'un coup — un ortho à la fois, monté depuis Drive.
- `Projet_mellifere/` est un dossier **partagé par le labo, en lecture seule** : le notebook n'y écrit jamais rien, toutes les sorties vont dans ton Drive perso (`OUTPUT_ROOT`).
- `HF_HUB_OFFLINE` est forcé à `1` par défaut dans `ortho_annotator` (poids locaux uniquement) : ce notebook le désactive explicitement pour pouvoir télécharger ViT-L depuis Hugging Face.

## 0. Runtime — choisir le GPU
Menu *Exécution > Modifier le type d'exécution*.

Le balayage dense traitait une fenêtre 784² à la fois (`dense()`, forward pass batch=1) — ça laissait la VRAM d'un L4/A100 largement inutilisée quel que soit le GPU choisi. **Corrigé** (`Embedder.dense_batch` + `DenseMatcher.score_batch`, `--embed-batch-size`, défaut 16) : le balayage encode plusieurs fenêtres par passage modèle. Choix maintenant réel :
- **L4** : bon rapport perf/unité de calcul Colab Pro, 24 Go permettent un batch confortable (32-64 à 784²) pour ce job. Choix par défaut de ce notebook.
- **A100** : plus rapide en temps mur si le budget d'unités de calcul n'est pas la contrainte — avec le batching, il se remplit correctement maintenant. À réserver si Maison en entier sur L4 s'avère trop long.
- **T4** : repli correct, batch un peu plus petit (16) recommandé (moins de VRAM, moins de cœurs tensoriels).
- **Pas de TPU** : DINOv3 via `transformers` n'a pas de chemin XLA propre, la mise en place coûterait plus cher que ce qu'elle ferait gagner ici.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Cloner le dépôt (privé — jeton GitHub à la volée, jamais écrit sur disque)
Un *fine-grained personal access token* avec accès lecture au dépôt `Lmague/benchmark-memoire` suffit. Laisser vide si le dépôt est public.

In [ ]:
import getpass, os

REPO = "Lmague/benchmark-memoire"
PROJECT_DIR = "/content/benchmark-memoire"

token = getpass.getpass(f"Jeton GitHub pour {REPO} (Entrée si public) : ")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 "{url}" "{PROJECT_DIR}"
else:
    print(f"{PROJECT_DIR} existe déjà, pas de re-clone.")
del token, url  # ne pas laisser le jeton en clair dans une variable notebook

## 2. Dépendances
`torch` est déjà présent (CUDA) sur Colab. Le reste (lecture géospatiale + `transformers`) ne l'est pas.

In [ ]:
!pip install -q rasterio fiona geopandas shapely pillow scipy "transformers>=4.40"

import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("GPU attribué :", name)
if not name:
    print("Pas de GPU — Exécution > Modifier le type d'exécution.")

## 3. Monter Drive et localiser `Projet_mellifere`
Recherche automatique sous `/content/drive` (fonctionne que le dossier soit dans *Mon Drive* ou ajouté comme raccourci depuis un partage) — **vérifier la sortie** avant de continuer, les chemins peuvent différer entre `MyDrive` et `Shareddrives`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import subprocess

hits = subprocess.run(
    ["find", "/content/drive", "-maxdepth", "5", "-iname", "Projet_mellifere", "-type", "d"],
    capture_output=True, text=True
).stdout.splitlines()
print("Candidats trouvés :")
for h in hits:
    print(" ", h)
if not hits:
    print("Aucun 'Projet_mellifere' trouvé automatiquement — fixer PROJET_ROOT à la main ci-dessous.")

In [ ]:
# À AJUSTER si la recherche ci-dessus a trouvé autre chose, ou plusieurs candidats.
PROJET_ROOT = hits[0] if hits else "/content/drive/MyDrive/Projet_mellifere"
print("PROJET_ROOT =", PROJET_ROOT)

!find "{PROJET_ROOT}/Orthomosaiques" -maxdepth 1 -type f | sort

## 4. Chemins et paramètres
`RASTER_MAISON` suppose la même arborescence/nommage que `Dataset_Leo`. **Corriger si la sortie de la cellule précédente montre un nom différent.**

In [ ]:
from pathlib import Path

EXISTING_ANNOTATIONS = f"{PROJET_ROOT}/Orthomosaiques/Annotations.gpkg"
RASTER_DIR = f"{PROJET_ROOT}/Orthomosaiques"
RASTER_MAISON = "Orthom_Maison_9Aout23_WGS84UTM18N.tif"  # cf. audit_dataset_leo.md — Lotcorn/Leuvul y sont exclusifs

# Projet_mellifere est un dossier PARTAGÉ par le labo Laliberté (lecture seule pour nous) :
# on n'y écrit jamais rien, ni les modèles appris ni les candidats. Toute sortie va dans
# TON Drive perso — à renommer/déplacer si tu préfères un autre chemin.
OUTPUT_ROOT = "/content/drive/MyDrive/mellifere_colab_runs"
RUN_DIR = f"{OUTPUT_ROOT}/session_vitl"
OUTPUT_GPKG = f"{RUN_DIR}/session_vitl.gpkg"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"

assert Path(EXISTING_ANNOTATIONS).is_file(), f"introuvable : {EXISTING_ANNOTATIONS}"
assert (Path(RASTER_DIR) / RASTER_MAISON).is_file(), f"introuvable : {RASTER_DIR}/{RASTER_MAISON}"
assert "Projet_mellifere" not in OUTPUT_ROOT, "ne rien écrire dans le dossier partagé du labo"
print("OK — chemins valides. Sorties dans :", RUN_DIR)

## 4b. Copier l'ortho localement (important pour le temps de calcul)
`Orthomosaiques/` est monté depuis Drive via FUSE : une lecture fenêtrée directe dessus (des centaines de petites lectures GDAL) peut être **beaucoup plus lente** qu'un disque local, largement plus que le calcul du modèle lui-même. Une copie séquentielle en une fois est nettement plus rapide qu'un accès aléatoire répété à travers la couche Drive. Maison fait ~14 Go — vérifier l'espace dispo (`!df -h /content`) si un doute.

In [ ]:
import shutil, time

LOCAL_RASTER_DIR = "/content/orthos"
Path(LOCAL_RASTER_DIR).mkdir(parents=True, exist_ok=True)

local_path = Path(LOCAL_RASTER_DIR) / RASTER_MAISON
if not local_path.is_file():
    t0 = time.time()
    shutil.copy(Path(RASTER_DIR) / RASTER_MAISON, local_path)
    print(f"Copié en {time.time() - t0:.0f} s -> {local_path}")
else:
    print(f"Déjà présent localement : {local_path}")

In [ ]:
import os

# ortho_annotator force HF_HUB_OFFLINE=1 par défaut (poids locaux uniquement, machine
# d'annotation sans réseau). Sur Colab on VEUT télécharger ViT-L : on désactive avant tout
# import du package (setdefault() ne touche pas une variable déjà positionnée).
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"

%cd {PROJECT_DIR}/tools/ortho_annotator
import sys
sys.path.insert(0, ".")

### Accès au modèle (dépôt "gated")
Les poids DINOv3 sur Hugging Face sont sous licence Meta à accepter explicitement — désactiver le mode hors-ligne ne suffit pas, un compte authentifié **avec accès accordé** est nécessaire, sinon 401 plus loin.
1. Compte Hugging Face (gratuit) si tu n'en as pas.
2. Ouvrir `huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m` connecté, cliquer *Agree and access repository* (une fois, ~immédiat).
3. `huggingface.co/settings/tokens` → créer un token (lecture seule suffit) → le coller ci-dessous.

In [ ]:
from huggingface_hub import login

hf_token = getpass.getpass("Jeton Hugging Face (lecture seule, après avoir accepté l'accès au modèle) : ")
login(token=hf_token)  # écrit le jeton dans ~/.cache/huggingface — repris par les sous-process `!python`
os.environ["HF_TOKEN"] = hf_token
del hf_token

## 5. Étape A — calibration ViT-L sur Maison
Mêmes `--span-m 5 --side-px 784` que les chiffres du README (ViT-S/16, 7 orthos) : seul le modèle change, pour isoler son effet. Coût attendu : quelques minutes sur GPU (5 espèces présentes sur Maison, `--windows-per-species 6` par défaut + fenêtres de calibration réservées).

In [ ]:
!python -m ortho_annotator prospect learn \
  --output "{OUTPUT_GPKG}" \
  --existing-annotations "{EXISTING_ANNOTATIONS}" \
  --raster-dir "{LOCAL_RASTER_DIR}" \
  --rasters "{RASTER_MAISON}" \
  --model "{MODEL_ID}" \
  --device cuda \
  --span-m 5.0 --side-px 784

In [ ]:
from pathlib import Path as _P

_dense_bank = _P(RUN_DIR) / "prospect" / "dense_tokens.npz"
if not _dense_bank.is_file():
    print("Banque de jetons denses absente — l'encodeur n'a probablement pas pu se charger "
          "(voir '== 2. Encodeur figé ==' dans le log ci-dessus : jeton HF manquant/invalide, "
          "accès au dépôt gated non accordé, ou pas de GPU). "
          "L'étage couleur seul a quand même tourné et donne des chiffres exploitables plus bas.")
    print("Étape B (--mode dense) échouera silencieusement (0 candidat) tant que ce n'est pas résolu.")
else:
    print("OK — banque de jetons denses présente :", _dense_bank)

In [ ]:
import json

cal = json.load(open(f"{RUN_DIR}/prospect/calibration.json"))

# Chiffres du README (ViT-S/16, 7 orthomosaïques, span 5 m / 784 px) — étage dense uniquement.
readme_dense = {
    "Ascsyr": (0.63, 0.35), "Daucar": (0.57, 0.50), "Eumac": (0.63, 0.50),
    "Lotcorn": (0.86, 0.61), "Solcan": (0.96, 0.78),
}

print(f"{'espèce':10s} {'ViT-S dense R/P (README, 7 orthos)':>36s}   {'ViT-L dense R/P (Maison seul)':>32s}")
for code, (r0, p0) in sorted(readme_dense.items()):
    d = cal.get("dense", {}).get(code)
    here = f"{d['recall']:.2f}/{d['precision']:.2f}" if d else "— (absent de Maison)"
    print(f"{code:10s} {r0:.2f}/{p0:.2f} {'':>28s}   {here:>32s}")

print("\nAttention : comparaison Maison-seul (ViT-L) vs 7-orthos (ViT-S) — indicatif, pas"
      " apples-to-apples. Si la tendance ne bouge pas nettement, ViT-L n'apporte probablement rien.")

### Comparaison propre : même site, seul le modèle change
La comparaison ci-dessus (ViT-L Maison-seul vs ViT-S 7-orthos README) est **confondue** : le détecteur couleur — qui ne dépend pas du modèle — donne lui aussi des R/P nettement meilleurs sur Maison seul que l'agrégat 7-orthos du README (ex. Ascsyr couleur 0,97 de rappel ici contre 0,05 dans le README). Une partie de ce qu'on attribuerait à ViT-L pourrait n'être qu'un effet de site (Maison plus facile, ou fenêtres réservées trop proches spatialement des fenêtres d'entraînement — points à 27-36 cm de médiane entre voisins). Pour isoler la taille de modèle : recalibrer en ViT-S sur le **même** Maison.

In [ ]:
RUN_DIR_S = f"{OUTPUT_ROOT}/session_vits"
OUTPUT_GPKG_S = f"{RUN_DIR_S}/session_vits.gpkg"
MODEL_ID_S = "facebook/dinov3-vits16-pretrain-lvd1689m"
Path(RUN_DIR_S).mkdir(parents=True, exist_ok=True)

!python -m ortho_annotator prospect learn \
  --output "{OUTPUT_GPKG_S}" \
  --existing-annotations "{EXISTING_ANNOTATIONS}" \
  --raster-dir "{LOCAL_RASTER_DIR}" \
  --rasters "{RASTER_MAISON}" \
  --model "{MODEL_ID_S}" \
  --device cuda \
  --span-m 5.0 --side-px 784

In [ ]:
cal_s = json.load(open(f"{RUN_DIR_S}/prospect/calibration.json"))
cal_l = json.load(open(f"{RUN_DIR}/prospect/calibration.json"))

codes = sorted(set(cal_s.get("dense", {})) | set(cal_l.get("dense", {})))
print(f"{'espèce':10s} {'ViT-S dense R/P (F1)':>22s}   {'ViT-L dense R/P (F1)':>22s}")
for code in codes:
    ds = cal_s.get("dense", {}).get(code)
    dl = cal_l.get("dense", {}).get(code)
    s_str = f"{ds['recall']:.2f}/{ds['precision']:.2f} ({ds['f1']:.2f})" if ds else "— (non calibré)"
    l_str = f"{dl['recall']:.2f}/{dl['precision']:.2f} ({dl['f1']:.2f})" if dl else "— (non calibré)"
    print(f"{code:10s} {s_str:>22s}   {l_str:>22s}")

print("\nSi le F1 ViT-L ne dépasse pas nettement le F1 ViT-S ici (même site, donc plus de "
      "confusion possible), ViT-L n'apporte rien pour ce job.")

## Porte de décision — résultat mesuré (Maison, même site, ViT-S vs ViT-L)

| espèce | ΔF1 dense (ViT-L − ViT-S) |
|---|---|
| Lotcorn | +0,28 |
| Ascsyr | +0,14 |
| Eumac | +0,09 |
| Solcan | +0,01 |

ViT-L gagne partout où c'est comparable — **le balayage complet est justifié**. Mais même avec ViT-L, dense ne bat couleur (F1) que pour **Ascsyr** (asclépiade défleurie, seule la texture/forme la trouve) ; couleur reste devant pour Eumac/Lotcorn/Solcan/Leuvul. D'où `--mode auto` plutôt que `--mode dense` ci-dessous : couleur pour les 4 espèces où elle gagne, dense+ViT-L seulement pour Ascsyr — décidé par le F1 mesuré, pas figé.

## 6. Étape B — balayage (`--mode auto`, couleur ou dense+ViT-L selon l'espèce)
`SMOKE_TEST=True` d'abord — mais attention, `--max-windows` ne borne QUE l'étage dense (`scan_raster_dense`). L'étage couleur (`scan_raster`) n'a pas cette limite et balaie toujours l'ortho en entier, quel que soit `SMOKE_TEST`. Concrètement : le smoke test donne déjà le résultat quasi final pour les espèces couleur (rapide, CPU), et seulement un aperçu tronqué pour les espèces dense/GPU. Passer à `SMOKE_TEST=False` ne refait donc que l'étage dense en entier (l'étage couleur recalcule à l'identique, un peu de temps perdu mais rien de faux).

In [ ]:
SMOKE_TEST = True
MAX_WINDOWS = 50 if SMOKE_TEST else 0  # 0 = pas de limite

# Fenêtres encodées ensemble par passage modèle. 16 = repli sûr (T4 y compris).
# Sur L4 (24 Go), monter à 32-64 pour un balayage complet nettement plus rapide.
EMBED_BATCH_SIZE = 16

# Un ortho à la fois : Lotcorn/Leuvul et le gros de Eumac/Solcan sont sur Maison ;
# ajouter TrailErable (Daucar) séparément une fois Maison validé.
RASTERS_TO_SCAN = [RASTER_MAISON]

for name in RASTERS_TO_SCAN:
    raster_path = f"{LOCAL_RASTER_DIR}/{name}"
    !python -m ortho_annotator prospect scan \
      --output "{OUTPUT_GPKG}" \
      --raster "{raster_path}" \
      --existing-annotations "{EXISTING_ANNOTATIONS}" \
      --mode auto \
      --model "{MODEL_ID}" \
      --device cuda \
      --embed-batch-size {EMBED_BATCH_SIZE} \
      --max-windows {MAX_WINDOWS}

Le résultat est déjà persisté dans `{RUN_DIR}/prospect/candidates.sqlite` (sur Drive) — pas de session Colab de plusieurs heures à protéger d'une déconnexion pour ce qui est déjà scanné. Le balayage n'est en revanche **pas repris fenêtre par fenêtre** : une déconnexion en cours d'ortho oblige à relancer cet ortho depuis le début (d'où : un ortho par cellule/tour de boucle, pas les 7 d'un coup).

## 7. Export QGIS + aperçu rapide

In [ ]:
CANDIDATES_GPKG = f"{RUN_DIR}/candidats_vitl.gpkg"
!python -m ortho_annotator prospect export \
  --output "{OUTPUT_GPKG}" \
  --dest "{CANDIDATES_GPKG}"

In [ ]:
from pathlib import Path as _P

if not _P(CANDIDATES_GPKG).is_file():
    raise SystemExit(
        "Aucun candidat exporté — remonter au log de l'Étape B : soit la banque de jetons "
        "denses était absente (voir la vérification après Étape A), soit tous les scores "
        "sont restés sous le seuil calibré sur ce sous-ensemble de fenêtres (SMOKE_TEST)."
    )
print("OK —", CANDIDATES_GPKG)

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

gdf = gpd.read_file(CANDIDATES_GPKG)
print(gdf["species"].value_counts())

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, column="species", markersize=4, legend=True, cmap="tab10")
ax.set_title("Candidats ViT-L dense — aperçu (ouvrir le .gpkg dans QGIS pour le détail)")
ax.set_aspect("equal")
plt.show()

## 8. Est-ce que ça marche vraiment ? — deux questions séparées

`calibration.json` (Étape A) répond à une seule question : **le détecteur généralise-t-il à des fenêtres neuves mais déjà vérifiées** (réservées, non utilisées pour construire les prototypes). Il ne dit rien sur les ~677K tuiles réellement balayées à l'Étape B — par définition, il n'existe aucune vérité terrain dessus, c'est tout l'intérêt de l'outil.

Deux vérifications complémentaires, pas une seule :

1. **Généralisation (déjà en main)** : relire `calibration.json`, en particulier `n_windows`/`n_points` par espèce avant de faire confiance au F1 — `Leuvul` (23 points annotés au total) aura un F1 mesuré sur une poignée de fenêtres, indicatif, pas fiable. `précision` reste un **minorant** : un candidat compté comme faux est parfois une vraie plante jamais pointée.
2. **Terrain réel (à faire après le balayage)** : échantillonner des candidats du balayage et les vérifier à l'œil. C'est la seule mesure qui porte sur la zone qui t'intéresse vraiment. Un échantillon **stratifié par score** (pas juste le top, qui ne montre que les cas faciles) donne une vraie précision, pas un minorant — cette fois tu regardes des tuiles jamais vérifiées, ton jugement EST la vérité terrain.

Le rappel sur la zone balayée ne se mesure pas par cet échantillonnage (on ne voit que ce qui a été proposé, jamais ce qui a été raté). Pour ça : choisir une petite sous-zone, l'annoter à la main de façon exhaustive, comparer au nombre de candidats que le balayage y a trouvés — coûteux mais c'est la seule façon honnête de mesurer un rappel sur du terrain neuf.

In [ ]:
import numpy as np

N_PER_BIN = 10   # candidats à vérifier par tranche de score, par espèce
N_BINS = 3       # tranches de score (facile / moyen / limite) — pas que le top

sampled = []
for species, group in gdf.groupby("species"):
    if len(group) == 0:
        continue
    bins = pd.qcut(group["score"], q=min(N_BINS, len(group)), duplicates="drop")
    for _, bin_group in group.groupby(bins, observed=True):
        n = min(N_PER_BIN, len(bin_group))
        sampled.append(bin_group.sample(n=n, random_state=0))

# pd.concat seul peut perdre le CRS après un groupby — on le reconstruit explicitement.
to_review = gpd.GeoDataFrame(pd.concat(sampled), crs=gdf.crs)
to_review = to_review.sort_values(["species", "score"], ascending=[True, False])
print(f"{len(to_review)} candidats à vérifier à l'œil (sur {len(gdf)} au total).")
print(to_review["species"].value_counts())

TO_REVIEW_GPKG = f"{RUN_DIR}/to_review_vitl.gpkg"
to_review.to_file(TO_REVIEW_GPKG, layer="a_verifier", driver="GPKG")
print("Écrit :", TO_REVIEW_GPKG)

Ouvrir `to_review_vitl.gpkg` dans QGIS par-dessus l'ortho (ou via `ortho_annotator serve` en local si tu as une copie du même fichier .tif), zoomer sur chaque point, compter vrai/faux par espèce. Le ratio obtenu **par tranche de score** te dit où placer un seuil de confiance pour la suite (ex. "au-dessus de ce score, j'accepte sans revérifier ; en dessous, revue manuelle systématique") — c'est plus utile qu'un seul chiffre global.